# Figure 4: HOMER versus TransBrain. Each method wins on the modality it encodes

TransBrain is the state of the art in mouse→human phenotype translation. It is a **transcriptomic**
translator. HOMER is a **connectional** one. Section 3 established that π carries connectional
organisation and not microstructure. This section applies the same principle to a competitor.

We made the prediction before looking: **TransBrain should lead on region identity**, which is largely a
molecular-identity question, and **HOMER should lead on connectional organisation**. That is what
happens.

| axis | HOMER | TransBrain | winner |
|---|---|---|---|
| region-level identity (AUROC) | 0.79 | **0.84** | TransBrain |
| principal-gradient translation (\|r\|) | **0.55** | 0.42 | HOMER |
| round-trip fidelity (mouse→human→mouse) | **0.98 / 0.95 / 0.97** | 0.89 / 0.82 / 0.83 | HOMER |
| prediction sharpness (effective targets) | **≈ 3** | ≈ 60 | HOMER |
| whole-brain coverage / absence detection | **yes** | no | HOMER |

This is **not** a competition we win. It is a statement of what each method encodes. Two caveats we state
up front:

- The region-identity benchmark is **TransBrain's own**, so its lead there has home advantage. Even so,
  the difference is **not significant** (paired Wilcoxon p = 0.17, n = 24 regions).
- We earlier claimed HOMER "localises better" than TransBrain. It does not. That apparent advantage was a
  **reduction artefact**: TransBrain's output is region-level, so comparing at parcel resolution flatters
  HOMER by construction. The panel was reframed as *sharpness/confidence*, which is a real difference.
  **Do not reinstate the localisation claim.**

> Rule: every number below is read from `transbrain_benchmark_summary.json` and
> `transbrain_roundtrip_maps.json`. An earlier draft carried "AUROC 0.85" and "leads 15 of 24", both of
> which are wrong (0.8446 → 0.84; the AUROC split is 16/8). Those numbers had been computed inside a
> figure script and never written to a file, which is how they drifted. `dump_benchmark_summary.py` now
> persists them.

In [ ]:
import sys, json, warnings
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, wilcoxon

warnings.filterwarnings('ignore')
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
LOGS = ROOT / 'outputs' / 'logs'

summ = json.loads((LOGS / 'transbrain_benchmark_summary.json').read_text())
bn   = json.loads((LOGS / 'transbrain_bn_distributions.json').read_text())
rt   = json.loads((LOGS / 'transbrain_roundtrip_maps.json').read_text())
tb   = json.loads((LOGS / 'transbrain_2025_benchmark.json').read_text())


def check(name, computed, expected, tol):
    ok = abs(computed - expected) <= tol
    print(f"  {'OK      ' if ok else 'MISMATCH'}  {name}: notebook {computed:.4f}  vs  log {expected:.4f}")
    assert ok, f'{name} diverged from the canonical log'


print(f"benchmark: {summ['n_benchmark_regions']} mouse regions scored as distributions over the "
      f"{summ['n_bn_regions']}-region Brainnetome atlas")
print(f"NOTE from the log: {summ['_note']}")

## 1. Region identity: TransBrain leads, and the lead is not significant (Fig. 4a, ED4a,b)

The benchmark: 24 literature-curated mouse regions, each scored as a distribution over the Brainnetome
atlas. AUROC asks whether the true human region ranks above the rest.

In [ ]:
h, t = summ['homer'], summ['transbrain']
print(f"{'metric':<24} {'HOMER':>8} {'TransBrain':>12}   winner")
print('-' * 60)
for k, label in [('top1', 'top-1'), ('top3', 'top-3'), ('top5', 'top-5'),
                 ('auroc', 'AUROC'), ('mass', 'mass in true region')]:
    win = 'TransBrain' if t[k] > h[k] else 'HOMER'
    print(f'{label:<24} {h[k]:>8.2f} {t[k]:>12.2f}   {win}')
print()
print(f"AUROC win split: TransBrain {summ['auroc_wins']['transbrain']} regions, "
      f"HOMER {summ['auroc_wins']['homer']} of {summ['n_benchmark_regions']}")
print(f"paired Wilcoxon on per-region AUROC: p = {summ['auroc_paired_wilcoxon_p']:.2f}  "
      f"({'n.s.' if summ['auroc_paired_wilcoxon_p'] >= 0.05 else 'significant'})")
print()
print('TransBrain leads region identity on its OWN benchmark, and the lead does not reach')
print('significance over 24 regions. HOMER nonetheless places MORE MASS on the correct region')
print(f"({h['mass']:.2f} vs {t['mass']:.2f}): it is right less often, but it commits to a narrower")
print('answer. That is the sharpness result below.')

### Verify the summary against the per-region distributions

The summary JSON exists because these numbers used to be computed *inside a figure script* and never
written down. We therefore recompute them from the raw per-region distributions and assert.

In [ ]:
def auroc(w, true):
    """Rank the true human region(s) against every other Brainnetome region."""
    w = np.asarray(w, float)
    pos, neg = w[list(true)], np.delete(w, list(true))
    return float(sum((p > q) + 0.5 * (p == q) for p in pos for q in neg) / (len(pos) * len(neg)))


def eff_n(w):
    """Effective number of target regions, 1 / sum p_i^2."""
    w = np.asarray(w, float)
    return float(1 / np.sum((w / w.sum()) ** 2)) if w.sum() > 0 else np.nan


regions = list(bn['regions'])
au_h = np.array([auroc(bn['regions'][r]['homer_w'], bn['regions'][r]['true']) for r in regions])
au_t = np.array([auroc(bn['regions'][r]['tb_w'], bn['regions'][r]['true']) for r in regions])
en_h = np.array([eff_n(bn['regions'][r]['homer_w']) for r in regions])
en_t = np.array([eff_n(bn['regions'][r]['tb_w']) for r in regions])

print(f'recomputed from the {len(regions)} per-region distributions:')
check('HOMER mean AUROC', float(au_h.mean()), h['auroc'], 1e-6)
check('TransBrain mean AUROC', float(au_t.mean()), t['auroc'], 1e-6)
check('TransBrain AUROC wins', float((au_t > au_h).sum()), summ['auroc_wins']['transbrain'], 0)
check('paired Wilcoxon p', float(wilcoxon(au_h, au_t).pvalue), summ['auroc_paired_wilcoxon_p'], 1e-6)
check('HOMER sharpness (eff. n)', float(np.nanmean(en_h)), h['eff_n'], 1e-4)
check('TransBrain sharpness (eff. n)', float(np.nanmean(en_t)), t['eff_n'], 1e-4)

In [ ]:
# ---------------- ED4b: per-region AUROC ----------------
order = np.argsort(au_t - au_h)
fig, ax = plt.subplots(figsize=(7.0, 6.4))
y = np.arange(len(regions))
ax.barh(y - 0.2, au_h[order], height=0.38, color='#1b4f8a', label='HOMER (connectional)')
ax.barh(y + 0.2, au_t[order], height=0.38, color='#e08a2b', label='TransBrain (transcriptomic)')
ax.axvline(0.5, color='0.4', ls=':', lw=1.2)
ax.text(0.505, len(regions) - 0.4, 'chance', fontsize=8, color='0.4')
ax.set_yticks(y)
ax.set_yticklabels([regions[i] for i in order], fontsize=7.5)
ax.set_xlabel('AUROC for the true human region')
ax.set_xlim(0, 1.0)
ax.legend(frameon=False, fontsize=9, loc='lower right')
ax.set_title('Region identity on TransBrain’s own benchmark\n'
             f"TransBrain {t['auroc']:.2f} vs HOMER {h['auroc']:.2f}, "
             f"wins {summ['auroc_wins']['transbrain']}/{summ['auroc_wins']['homer']}, "
             f"paired Wilcoxon p = {summ['auroc_paired_wilcoxon_p']:.2f} (n.s.)",
             fontweight='bold', loc='left', fontsize=10.5)
for s in ('top', 'right'):
    ax.spines[s].set_visible(False)
plt.show()

## 2. Sharpness, the real difference (Fig. 4a right, ED4c)

We measure the effective number of target regions, $1/\sum_i p_i^2$. A method that spreads its answer
over 60 regions has told you much less than one that commits to 3, *even if the 60-way answer ranks the
true region slightly higher*.

> The published panel 4a once reported sharpness **64**. The true value is **60.1**. That panel had
> **no source script at all**; it was an orphaned PNG. `fig4/make_panelA_accuracy.py` now builds it.

In [ ]:
print(f"effective number of target regions (1 / sum p_i^2):")
print(f"  HOMER       {h['eff_n']:>6.1f}")
print(f"  TransBrain  {t['eff_n']:>6.1f}")
print(f"  ratio       {t['eff_n'] / h['eff_n']:>6.1f}x broader")
print()

# ---------------- Fig 4a ----------------
fig, axes = plt.subplots(1, 2, figsize=(8.0, 4.0))
cols = ['#1b4f8a', '#e08a2b']

axes[0].bar(['HOMER', 'TransBrain'], [h['auroc'], t['auroc']], color=cols, width=0.55)
axes[0].axhline(0.5, color='0.5', ls=':', lw=1)
axes[0].set_ylim(0, 1.0)
axes[0].set_ylabel('AUROC for the true human region')
for i, v in enumerate([h['auroc'], t['auroc']]):
    axes[0].text(i, v + 0.02, f'{v:.2f}', ha='center', fontsize=9.5)
axes[0].set_title(f"Region identity\nTransBrain leads (p = {summ['auroc_paired_wilcoxon_p']:.2f}, n.s.)",
                  fontweight='bold', loc='left', fontsize=10)

axes[1].bar(['HOMER', 'TransBrain'], [h['eff_n'], t['eff_n']], color=cols, width=0.55)
axes[1].set_yscale('log')
axes[1].set_ylabel('effective no. of target regions (1/Σpᵢ²)')
for i, v in enumerate([h['eff_n'], t['eff_n']]):
    axes[1].text(i, v * 1.15, f'{v:.1f}', ha='center', fontsize=9.5)
axes[1].set_title(f"Sharpness\nHOMER commits to ≈{h['eff_n']:.0f} regions, TransBrain to ≈{t['eff_n']:.0f}",
                  fontweight='bold', loc='left', fontsize=10)

for a in axes:
    for s in ('top', 'right'):
        a.spines[s].set_visible(False)
fig.subplots_adjust(wspace=0.35, top=0.80)
plt.show()

## 3. The connectional gradient: HOMER leads (ED4d)

The §3 result, run head-to-head. Translate the mouse principal gradient forward with each method and
correlate with the observed human gradient, over the 60 Brainnetome regions both methods cover.

In [ ]:
g = tb['head_to_head_gradient']
for k, v in g.items():
    if isinstance(v, (int, float)):
        print(f'  {k:36s} {v:.2f}')
print()
print('HOMER translates the connectional gradient more faithfully than TransBrain. Section 3 predicted')
print('this: the gradient IS connectional organisation, and that is the modality pi encodes.')
print()
print('The two translated gradients also agree with each other strongly. The methods are not')
print('contradicting each other; they are resolving the same axis with different fidelity.')

## 4. Round-trip fidelity (Fig. 4b): HOMER's clearest advantage

Translate a mouse phenotype mouse→human→mouse and correlate the recovered map with the original. A
faithful correspondence should be close to invertible.

Three phenotypes: the principal FC gradient (smooth), an optogenetic agranular-insula circuit (focal),
and the **Magel2** autism-model atrophy pattern (focal, disease-relevant).

> ⚠️ **A bug found while building this notebook (fixed 2026-07-13).** `dump_roundtrip_maps.py` used to
> score HOMER over the **52** mouse regions its parcellation covers, but TransBrain over all **68**
> regions in `Config.MOUSE_REGIONS`. For the gradient, 16 of those 68 had no measured value and were
> **mean-filled** before scoring. The two numbers were not comparable, and the error did not push
> consistently in one direction: it inflated TransBrain on the optogenetic map (0.82 → 0.91) and
> deflated it on the gradient (0.89 → 0.87).
>
> Both methods are now scored on the **identical 52 regions** where the phenotype is measured. HOMER
> still leads all three. The published TransBrain values change from 0.87 / 0.91 / 0.81 to
> **0.89 / 0.82 / 0.83**.
>
> The general lesson, for the third time in this repo: **a head-to-head must be computed on the same
> domain with the same estimator.** If two numbers sit side by side in a table, check that they were made
> the same way.

In [ ]:
print(f"{'phenotype':<28} {'HOMER':>8} {'TransBrain':>12}   {'margin':>8}")
print('-' * 62)
names = {'gradient': 'principal FC gradient', 'AI_opto': 'agranular-insula opto', 'Magel2': 'Magel2 (autism model)'}
for k, label in names.items():
    d = rt[k]
    print(f"{label:<28} {d['r_homer']:>8.2f} {d['r_transbrain']:>12.2f}   "
          f"{d['r_homer'] - d['r_transbrain']:>+8.2f}")
print()
print(f"scored over the same {rt['gradient']['n_regions_scored']} regions for both methods")
print()
print('The margin is NARROWEST on the smooth principal gradient and WIDEST on the two FOCAL maps.')
print('That is the sharpness result again: region-level smoothing destroys the spatial detail a focal')
print('phenotype lives in, so the coarser the output, the more the round trip loses. The gradient is')
print('smooth enough that region-level averaging costs comparatively little.')
print()
print('Note what we do NOT say: an earlier draft claimed the margin was widest on Magel2 specifically.')
print('On the corrected, matched scoring the optogenetic map ties it. The mechanism survives; the')
print('Magel2-specific version of the claim does not.')

In [ ]:
# ---------------- Fig 4b: round-trip scatters + error ----------------
fig, axes = plt.subplots(1, 3, figsize=(11.0, 3.7))
def corr(a, b):
    m = np.isfinite(a) & np.isfinite(b)
    return float(pearsonr(a[m], b[m])[0])


# The headline r is REGION-LEVEL, not parcel-level: the stored arrays are per-parcel (1,864 long),
# but both methods are scored after averaging within each mouse region. Aggregate before correlating,
# or you will get a different number (0.974 instead of 0.977) and think something is broken.
mm = json.loads((ROOT / 'data_external' / 'mouse_sc_meta.json').read_text())
acr = np.array([mm['structure_acronyms'][i] for i in mm['node_struct_idx']])

for ax, (k, label) in zip(axes, names.items()):
    d = rt[k]
    orig = np.array(d['original'], float)
    hm = np.array(d['homer'], float)
    tbm = np.array(d['transbrain'], float)
    scored = d['regions_scored']                    # the identical 52 regions, both methods

    o_r = np.array([np.nanmean(orig[acr == a]) for a in scored])
    h_r = np.array([np.nanmean(hm[acr == a]) for a in scored])
    t_r = np.array([np.nanmean(tbm[acr == a]) for a in scored])
    check(f'{label} HOMER r', corr(o_r, h_r), d['r_homer'], 1e-6)
    check(f'{label} TransBrain r', corr(o_r, t_r), d['r_transbrain'], 1e-6)

    ax.scatter(o_r, h_r, s=26, alpha=0.75, color='#1b4f8a',
               label=f"HOMER r = {d['r_homer']:.2f}")
    ax.scatter(o_r, t_r, s=26, alpha=0.75, color='#e08a2b',
               label=f"TransBrain r = {d['r_transbrain']:.2f}")
    lim = [o_r.min(), o_r.max()]
    ax.plot(lim, lim, color='0.5', ls='--', lw=1)
    ax.set_xlabel('original mouse map (region mean)')
    ax.set_ylabel('recovered after mouse→human→mouse')
    ax.legend(frameon=False, fontsize=8, loc='upper left')
    ax.set_title(f"{label}  ({len(scored)} regions)", fontweight='bold', loc='left', fontsize=10)
    for s_ in ('top', 'right'):
        ax.spines[s_].set_visible(False)
fig.subplots_adjust(wspace=0.38, top=0.82)
fig.suptitle('Round-trip fidelity: HOMER recovers what it sent; the margin widens on the focal maps',
             fontsize=11.5, fontweight='bold', x=0.02, ha='left', y=1.04)
plt.show()

## 5. What we are NOT claiming: the localisation artefact

**We once claimed HOMER "localises better" than TransBrain. That was wrong.**

TransBrain's output is region-level (~120 Brainnetome regions). HOMER's is parcel-level (2,094 parcels).
Score a localisation metric at parcel resolution and HOMER wins **by construction**: the comparison
measures the output granularity rather than the mapping quality. It was a reduction artefact.

Panel 4c was therefore reframed. It now shows a difference that is real: HOMER returns a **full
per-parcel distribution** with a per-prediction confidence grade, and TransBrain returns a region-level
answer. That is a difference in *what the output is*. It is not a claim that one is more accurate.

**Do not reinstate the localisation claim.** If a future panel appears to show HOMER localising better,
check the resolution of the two outputs first.

## 6. Summary: complementary instruments rather than competitors

| capability | HOMER | TransBrain |
|---|---|---|
| region-level accuracy | ◐ 0.79 | ● 0.84 (n.s.) |
| smooth-gradient translation | ● 0.55 | ◐ 0.42 |
| spatial resolution | ● parcel (2,094) | ◐ region (~120) |
| prediction sharpness | ● ≈ 3 targets | ○ ≈ 60 targets |
| per-prediction confidence | ● evidence tiers | ○ none |
| round-trip fidelity | ● 0.95–0.97 | ◐ 0.81–0.91 |
| whole-brain coverage / absence | ● reports where no homologue exists | ○ none |

**A transcriptomic translator is the better instrument for region identity. A connectional one is the
better instrument for connectional organisation.** Neither method is defective here. This is what §3
predicts, applied to a competitor. The two occupy distinct niches: region-level phenotype transfer versus
calibrated whole-brain correspondence.

The last row is the bridge to §5. HOMER can say *where no homologue exists*. TransBrain cannot give that
answer, and it turns out to be the answer that predicts disease.

### Panels not produced here

Fig. 4c (the VISp/MOp/CA2 resolution brains) and the glass-brain columns of 4b are volumetric renderings:
`fig4/make_panelC_candidates.py` and `fig4/make_panelB_roundtrip.py`. Fig. 4d (the capability matrix) is
`fig4/make_panelD_matrix.py`.